# VieNeu-TTS v3 Turbo: Vietnamese text-to-speech

- Model: [`pnnbao-ump/VieNeu-TTS-v3-Turbo`](https://huggingface.co/pnnbao-ump/VieNeu-TTS-v3-Turbo) through the `vieneu` package ([GitHub](https://github.com/pnnbao97/VieNeu-TTS)).
- Before you start: **Runtime › Change runtime type › GPU** (T4 or L4 is enough).
- Steps:
  1. Install and load the model.
  2. (Optional) Upload a reference audio clip to clone a voice (3–8 s, clear speech).
  3. Enter a long text, pick a voice (preset or clone), then press **Generate**.
- Output is a 48 kHz WAV file in `/content/outputs` (Files panel on the left).

## 1. Install

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!uv pip install --system "vieneu[cuda]==3.8.3" torchvision==0.23.0

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Load the model

In [2]:
import time
from pathlib import Path

import ipywidgets as widgets
import torch
from google.colab import files
from IPython.display import Audio, display
from vieneu import Vieneu

OUTPUT_DIR = Path("/content/outputs")
REF_DIR = Path("/content/ref_voices")
OUTPUT_DIR.mkdir(exist_ok=True)
REF_DIR.mkdir(exist_ok=True)

print(f"torch {torch.__version__}, CUDA {torch.version.cuda}, GPU {torch.cuda.get_device_name(0)}")

tts = Vieneu(device="cuda")
presets = tts.list_preset_voices()
print(f"Model loaded, {len(presets)} preset voices")

torch 2.8.0+cu128, CUDA 12.8, GPU Tesla T4


config.json:   0%|          | 0.00/2.15k [00:00<?, ?B/s]

[transformers] You are using a model of type `vieneu_v3` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/8.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/22.3k [00:00<?, ?B/s]

...

speaker_encoder.onnx: reconstructing file:   0%|          |  0.00B / 28.3MB            

speaker_encoder.onnx: downloading bytes:           |  0.00B            

denoiser.onnx: reconstructing file:   0%|          |  0.00B / 42.7MB            

denoiser.onnx: downloading bytes:           |  0.00B            

Model loaded, 25 preset voices


## 3. (Optional) Upload a voice to clone

- Upload one or more `.wav` / `.mp3` files, each 3–8 s long.
- Re-run step 4 afterwards so the new voice shows up in the list.

In [9]:
for name, data in files.upload().items():
    path = REF_DIR / name
    path.write_bytes(data)
    print(f"Saved {path}")

Saving Gió đêm thổi qu-695cb6291f_b6291f.mp3 to Gió đêm thổi qu-695cb6291f_b6291f.mp3
Saved /content/ref_voices/Gió đêm thổi qu-695cb6291f_b6291f.mp3


## 4. Generate speech

In [10]:
def voice_options():
    options = [(label, ("preset", voice_id)) for label, voice_id in presets]
    options += [(f"🎤 Clone: {path.name}", ("clone", str(path))) for path in sorted(REF_DIR.iterdir())]
    return options

INFER_OPTIONS = {
    "temperature": 0.65,
    "top_k": 15,
    "top_p": 0.9,
    "repetition_penalty": 1.2,
    "max_chars": 200,
    "apply_watermark": True,
}

def synthesize(text, voice, denoise):
    kind, value = voice
    if kind == "clone":
        return tts.infer(text, ref_audio=value, denoise=denoise, **INFER_OPTIONS)
    return tts.infer(text, voice=value, **INFER_OPTIONS)


text_input = widgets.Textarea(placeholder="Enter a long text...", layout=widgets.Layout(width="100%", height="240px"))
voice_input = widgets.Dropdown(options=voice_options(), description="Voice", layout=widgets.Layout(width="100%"))
denoise_input = widgets.Checkbox(value=True, description="Denoise the reference audio (clone only)")
generate_button = widgets.Button(description="Generate", button_style="primary", icon="play")
output = widgets.Output()


def on_generate(_):
    output.clear_output()
    with output:
        text = text_input.value.strip()
        if not text:
            print("Please enter some text.")
            return
        generate_button.disabled = True
        started = time.time()
        try:
            print(f"Generating {len(text)} characters...")
            audio = synthesize(text, voice_input.value, denoise_input.value)
            path = OUTPUT_DIR / f"speech_{time.strftime('%Y%m%d_%H%M%S')}.wav"
            tts.save(audio, str(path))
            print(f"Done in {time.time() - started:.1f}s → {path}")
            display(Audio(str(path)))
        finally:
            generate_button.disabled = False


generate_button.on_click(on_generate)
display(widgets.VBox([text_input, voice_input, denoise_input, generate_button, output]))

## 5. (Optional) Download every generated file

In [ ]:
!cd /content && zip -qr outputs.zip outputs
files.download("/content/outputs.zip")